# TM358 TMA03 22J data processing


__Purpose:__ compare ANN with random forrest 

## Upload libraries


In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression 
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
from sklearn import model_selection
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from math import sqrt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score


from keras.layers import Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.layers import Dropout
from keras.utils.np_utils import to_categorical

## upload datasets and preprocess data


In [2]:
pm = pd.read_csv('data/Air Quality Benchmark dataset.csv')
pm.head()

,Time_stamp,boxName,PM 2.5,temp,pressure,humidity,wind_speed,Time of Day,Peak/NoPeak,Day,Week Day,Weather,Weather Description,label
0,2018-12-31 18:30:12,iGude,18.20,7.71,1032,100,3.09,Evening_Hours,Peak,Monday,Workday,Clouds,broken clouds,normal
1,2018-12-31 18:32:41,iGude,19.27,7.71,1032,100,3.09,Evening_Hours,Peak,Monday,Workday,Clouds,broken clouds,normal
2,2018-12-31 18:35:11,iGude,18.57,7.71,1032,100,3.09,Evening_Hours,Peak,Monday,Workday,Clouds,broken clouds,normal
3,2018-12-31 18:37:41,iGude,17.85,7.71,1032,100,3.09,Evening_Hours,Peak,Monday,Workday,Clouds,broken clouds,normal
4,2018-12-31 18:40:11,iGude,25.95,7.71,1032,100,3.09,Evening_Hours,Peak,Monday,Workday,Clouds,broken clouds,normal


In [3]:
pm.describe()

,PM 2.5,temp,pressure,humidity,wind_speed
count,1.230693e+06,1.230693e+06,1.230693e+06,1.230693e+06,1.230693e+06
mean,8.343975e+00,9.813300e+00,1.015272e+03,7.533406e+01,3.781297e+00
std,1.244952e+01,7.726966e+00,1.068135e+01,1.876159e+01,2.313942e+00
min,0.000000e+00,-8.910000e+00,9.760000e+02,1.400000e+01,3.100000e-01
25%,2.470000e+00,4.140000e+00,1.009000e+03,6.500000e+01,2.100000e+00
50%,5.000000e+00,8.200000e+00,1.015000e+03,8.000000e+01,3.600000e+00
75%,1.090000e+01,1.469000e+01,1.022000e+03,9.200000e+01,5.100000e+00
max,9.999000e+02,3.928000e+01,1.046000e+03,1.000000e+02,1.750000e+01


In [4]:
# set list of category of all categorical features.
Time_of_day_category=pm['Time of Day'].unique()
peak_category=pm['Peak/NoPeak'].unique()
Day_category=pm['Day'].unique()
Weekday_category=pm['Week Day'].unique()
Weather_category=pm['Weather'].unique()
list_of_category_list=[Time_of_day_category,peak_category,Day_category,Weekday_category,Weather_category]
print(list_of_category_list)

[array(['Evening_Hours', 'Night_Hours', 'Morning_Hours', 'Afternoon_Hours'],
      dtype=object), array(['Peak', 'No_Peak'], dtype=object), array(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday',
       'Sunday'], dtype=object), array(['Workday', 'Weekend'], dtype=object), array(['Clouds', 'Mist', 'Fog', 'Rain', 'Drizzle', 'Drizzle + Rain',
       'Clear', 'Snow', 'Snow + Mist', 'Drizzle + Snow + Mist',
       'Drizzle + Rain + Mist', 'Drizzle + Mist', 'Rain + Snow',
       'Rain + Mist', 'Mist + Fog', 'Thunderstorm + Rain', 'Squall',
       'Haze', 'Thunderstorm', 'Thunderstorm + Squall', 'Rain + Fog',
       'Snow + Mist + Fog', 'Snow + Fog'], dtype=object)]


In [5]:
# set list of dictionaries for each categorical features
Time_of_day_dict={}
peak_dict={}
Day_dict={}
Weekday_dict={}
Weather_dict={}
list_of_dict=[Time_of_day_dict,peak_dict,Day_dict,Weekday_dict,Weather_dict]

for j in range(len(list_of_dict)):
    for i,items in enumerate(list_of_category_list[j]):
        list_of_dict[j][i]=items
list_of_dict

[{0: 'Evening_Hours',
  1: 'Night_Hours',
  2: 'Morning_Hours',
  3: 'Afternoon_Hours'},
 {0: 'Peak', 1: 'No_Peak'},
 {0: 'Monday',
  1: 'Tuesday',
  2: 'Wednesday',
  3: 'Thursday',
  4: 'Friday',
  5: 'Saturday',
  6: 'Sunday'},
 {0: 'Workday', 1: 'Weekend'},
 {0: 'Clouds',
  1: 'Mist',
  2: 'Fog',
  3: 'Rain',
  4: 'Drizzle',
  5: 'Drizzle + Rain',
  6: 'Clear',
  7: 'Snow',
  8: 'Snow + Mist',
  9: 'Drizzle + Snow + Mist',
  10: 'Drizzle + Rain + Mist',
  11: 'Drizzle + Mist',
  12: 'Rain + Snow',
  13: 'Rain + Mist',
  14: 'Mist + Fog',
  15: 'Thunderstorm + Rain',
  16: 'Squall',
  17: 'Haze',
  18: 'Thunderstorm',
  19: 'Thunderstorm + Squall',
  20: 'Rain + Fog',
  21: 'Snow + Mist + Fog',
  22: 'Snow + Fog'}]

In [6]:
# set list of dictionaries for each categorical features
Time_of_day_dict={}
peak_dict={}
Day_dict={}
Weekday_dict={}
Weather_dict={}
list_of_dict=[Time_of_day_dict,peak_dict,Day_dict,Weekday_dict,Weather_dict]

for j in range(len(list_of_dict)):
    for i,items in enumerate(list_of_category_list[j]):
        list_of_dict[j][i]=items
list_of_dict

[{0: 'Evening_Hours',
  1: 'Night_Hours',
  2: 'Morning_Hours',
  3: 'Afternoon_Hours'},
 {0: 'Peak', 1: 'No_Peak'},
 {0: 'Monday',
  1: 'Tuesday',
  2: 'Wednesday',
  3: 'Thursday',
  4: 'Friday',
  5: 'Saturday',
  6: 'Sunday'},
 {0: 'Workday', 1: 'Weekend'},
 {0: 'Clouds',
  1: 'Mist',
  2: 'Fog',
  3: 'Rain',
  4: 'Drizzle',
  5: 'Drizzle + Rain',
  6: 'Clear',
  7: 'Snow',
  8: 'Snow + Mist',
  9: 'Drizzle + Snow + Mist',
  10: 'Drizzle + Rain + Mist',
  11: 'Drizzle + Mist',
  12: 'Rain + Snow',
  13: 'Rain + Mist',
  14: 'Mist + Fog',
  15: 'Thunderstorm + Rain',
  16: 'Squall',
  17: 'Haze',
  18: 'Thunderstorm',
  19: 'Thunderstorm + Squall',
  20: 'Rain + Fog',
  21: 'Snow + Mist + Fog',
  22: 'Snow + Fog'}]

In [7]:
pm.replace(to_replace= r'^\s*$', value=np.nan,regex=True, inplace=True ) 
#replace any unit value that only contains " " or space, with 0.
pm.isnull().any() 
#check whether each column contains a missing value

Time_stamp             False
boxName                False
PM 2.5                 False
temp                   False
pressure               False
humidity               False
wind_speed             False
Time of Day            False
Peak/NoPeak            False
Day                    False
Week Day               False
Weather                False
Weather Description    False
label                  False
dtype: bool

In [8]:
# after finding the longtitude and latitude values for each sensors, we store them into a dictionary.
place_dict = {
    'iGude':[50.100216, 8.693827],
    'Rothschildallee':[50.127313, 8.696384],
    'FeinstaubFFM':[50.128178, 8.691848],
    'Frankfurt_Riederwald':[50.128679, 8.732761],
    'Medienzentrum Frankfurt':[50.113110, 8.685966],
    'FFM_Westend_Sued':[50.115127, 8.658714],
    'ioki':[50.117625, 8.671350],
    'Ginnheim_Dust_Light_Temp':[50.143277, 8.647729],
    'Alt Bornheim Feinstaub':[50.130067, 8.710821],
    'Bernem':[50.124220, 8.709344],
    'MousonSense':[50.147694, 8.695171]
}

# we also drop those rows with sensors that cannot identify its location.
drop_rows = ['nordsand', 's4', 'Luftdaten.info [6703181]']
for i in drop_rows:
    pm = pm.drop(pm[pm['boxName'] == i].index)

In [9]:
# Convert each text value in categorical feature to numerical value
columns=['Time of Day','Peak/NoPeak','Day','Week Day','Weather','label']

for columns in columns:
    u = pm[columns].unique()
    
    def conver(x):
        return np.argwhere(u==x)[0,0]
    
    pm[columns] = pm[columns].map(conver)

# using a loop to append latitude and longtitude for each observation at the end of dataset.
for k, v in place_dict.items():
    pm.loc[pm['boxName'] == k, 'latitude'] = v[0]
    pm.loc[pm['boxName'] == k, 'longitude'] = v[1]

In [10]:
# convert the type of Time_stamp(string) into Timestamp, 
#so that we can straightly use days for that time stamp in the following step.
pm['Time_stamp'] = pd.to_datetime(pm['Time_stamp'])
type(pm.iloc[0]['Time_stamp'])

# Split dataframe by boxName
classification=list(pm['boxName'].unique())
for i in classification:
    pm1=pm[pm['boxName'].isin([i])]
    exec("group%s=pm1"%classification.index(i))

# Process multiple DataFrame data in batches
name=[]
data=[]
for j in range(0,len(classification)):
    dfName='group'+str(j)
    dfData=eval(dfName)
    name.append(dfName)
    data.append(dfData)
data[3].head() # show the splited dataframe where the boxName is Frankfurt_Riederwold.

,Time_stamp,boxName,PM 2.5,temp,pressure,humidity,wind_speed,Time of Day,Peak/NoPeak,Day,Week Day,Weather,Weather Description,label,latitude,longitude
404500,2018-12-31 18:31:04,FeinstaubFFM,16.62,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.128178,8.691848
404501,2018-12-31 18:33:32,FeinstaubFFM,17.77,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.128178,8.691848
404502,2018-12-31 18:36:01,FeinstaubFFM,19.00,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,1,50.128178,8.691848
404503,2018-12-31 18:38:29,FeinstaubFFM,19.02,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,1,50.128178,8.691848
404504,2018-12-31 18:40:59,FeinstaubFFM,18.90,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.128178,8.691848


From: https://www.kaggle.com/code/patriciaryserwelch/notebookef03872adc/edit

In order to apply classification model in this dataset, we first need to change our target PM 2.5 which is continous, into categorical. From World Health Organization (WHO) we find out the air quality guidelines and interim targets for PM 2.5:

Interim target-1(IT-1): PM 2.5 = 35 ug/m^3 It is associated with about a 15% gigher long-term mortality risk relative to AQG level.

Interim target-2(IT-2): PM 2.5 = 25 ug/m^3 In addition to other health benefits, this lebel lower the risk of premature mortality bu approximately 6% [2-11%] relative to the IT-1 level.

Interim target-3(IT-3): PM 2.5 = 15 ug/m^3 In addition to other health benefits, this level reduced the mortality risk by approximately 6% [2-11%] relative to the -IT-2 level.

Air quality guideline (AQG): PM 2.5 = 10 ug/m^3 


These are the lowest levels at which total, cardiopulmonary and lung cancer mortality have been shown to increase with more than 95% confidence in response to long-term exposure to PM2.5

In [11]:
pm.loc[(pm['PM 2.5'] >= 0)  & (pm['PM 2.5'] <= 10),  'PM 2.5 level'] = "AQG"
pm.loc[(pm['PM 2.5'] > 10)  & (pm['PM 2.5'] <= 15),  'PM 2.5 level'] = "IT-3"
pm.loc[(pm['PM 2.5'] > 15)  & (pm['PM 2.5'] <= 25),  'PM 2.5 level'] = "IT-2"
pm.loc[(pm['PM 2.5'] > 25)  & (pm['PM 2.5'] <= 35),  'PM 2.5 level'] = "IT-1"
pm.loc[(pm['PM 2.5'] > 35), 'PM 2.5 level'] = "exceed standard"
pm.head()







,Time_stamp,boxName,PM 2.5,temp,pressure,humidity,wind_speed,Time of Day,Peak/NoPeak,Day,Week Day,Weather,Weather Description,label,latitude,longitude,PM 2.5 level
0,2018-12-31 18:30:12,iGude,18.20,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2
1,2018-12-31 18:32:41,iGude,19.27,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2
2,2018-12-31 18:35:11,iGude,18.57,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2
3,2018-12-31 18:37:41,iGude,17.85,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2
4,2018-12-31 18:40:11,iGude,25.95,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-1


In [12]:
pm.loc[(pm['PM 2.5'] >= 0)  & (pm['PM 2.5'] <= 10),  'PM 2.5 level num'] = 1
pm.loc[(pm['PM 2.5'] > 10)  & (pm['PM 2.5'] <= 15),  'PM 2.5 level num'] = 2
pm.loc[(pm['PM 2.5'] > 15)  & (pm['PM 2.5'] <= 25),  'PM 2.5 level num'] = 3
pm.loc[(pm['PM 2.5'] > 25)  & (pm['PM 2.5'] <= 35),  'PM 2.5 level num'] = 4
pm.loc[(pm['PM 2.5'] > 35), 'PM 2.5 level num'] = 5
pm['PM 2.5 level num']

0          3.0
1          3.0
2          3.0
3          3.0
4          4.0
5          4.0
6          4.0
7          3.0
8          3.0
9          3.0
10         3.0
11         3.0
12         3.0
13         3.0
14         3.0
15         3.0
16         3.0
17         3.0
18         3.0
19         3.0
20         3.0
21         3.0
22         3.0
23         3.0
24         3.0
25         3.0
26         3.0
27         3.0
28         3.0
29         3.0
          ... 
1230663    1.0
1230664    1.0
1230665    1.0
1230666    1.0
1230667    1.0
1230668    1.0
1230669    1.0
1230670    1.0
1230671    1.0
1230672    1.0
1230673    1.0
1230674    1.0
1230675    1.0
1230676    1.0
1230677    1.0
1230678    1.0
1230679    1.0
1230680    1.0
1230681    1.0
1230682    1.0
1230683    1.0
1230684    1.0
1230685    1.0
1230686    1.0
1230687    1.0
1230688    1.0
1230689    1.0
1230690    1.0
1230691    1.0
1230692    1.0
Name: PM 2.5 level num, Length: 1002953, dtype: float64

In [13]:
a = pm.to_csv('data/processed_data.csv', index=False)
a

## check dataset

In [14]:
data = pd.read_csv('data/processed_data.csv')
data.head()

,Time_stamp,boxName,PM 2.5,temp,pressure,humidity,wind_speed,Time of Day,Peak/NoPeak,Day,Week Day,Weather,Weather Description,label,latitude,longitude,PM 2.5 level,PM 2.5 level num
0,2018-12-31 18:30:12,iGude,18.20,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2,3.0
1,2018-12-31 18:32:41,iGude,19.27,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2,3.0
2,2018-12-31 18:35:11,iGude,18.57,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2,3.0
3,2018-12-31 18:37:41,iGude,17.85,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-2,3.0
4,2018-12-31 18:40:11,iGude,25.95,7.71,1032,100,3.09,0,0,0,0,0,broken clouds,0,50.100216,8.693827,IT-1,4.0


In [15]:
data.describe()

,PM 2.5,temp,pressure,humidity,wind_speed,Time of Day,Peak/NoPeak,Day,Week Day,Weather,label,latitude,longitude,PM 2.5 level num
count,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06,1.002953e+06
mean,8.416614e+00,9.612243e+00,1.015361e+03,7.498158e+01,3.800234e+00,1.497043e+00,6.247671e-01,2.996509e+00,2.838239e-01,1.675201e+00,4.798729e-02,5.012928e+01,8.702038e+00,1.509053e+00
std,1.235792e+01,7.662711e+00,1.077643e+01,1.878453e+01,2.327448e+00,9.987756e-01,4.841832e-01,1.997420e+00,4.508526e-01,2.667053e+00,2.137395e-01,8.173603e-03,2.212469e-02,9.393717e-01
min,0.000000e+00,-8.910000e+00,9.760000e+02,1.400000e+01,3.100000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.010022e+01,8.647729e+00,1.000000e+00
25%,2.500000e+00,4.060000e+00,1.009000e+03,6.400000e+01,2.100000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.012731e+01,8.691848e+00,1.000000e+00
50%,5.100000e+00,7.990000e+00,1.015000e+03,8.000000e+01,3.600000e+00,1.000000e+00,1.000000e+00,3.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.012818e+01,8.696384e+00,1.000000e+00
75%,1.103000e+01,1.434000e+01,1.022000e+03,9.200000e+01,5.100000e+00,2.000000e+00,1.000000e+00,5.000000e+00,1.000000e+00,3.000000e+00,0.000000e+00,5.012868e+01,8.709344e+00,2.000000e+00
max,9.675300e+02,3.928000e+01,1.046000e+03,1.000000e+02,1.750000e+01,3.000000e+00,1.000000e+00,6.000000e+00,1.000000e+00,2.200000e+01,1.000000e+00,5.014769e+01,8.732761e+00,5.000000e+00


In [16]:
data.columns

Index(['Time_stamp', 'boxName', 'PM 2.5', 'temp', 'pressure', 'humidity',
       'wind_speed', 'Time of Day', 'Peak/NoPeak', 'Day', 'Week Day',
       'Weather', 'Weather Description', 'label', 'latitude', 'longitude',
       'PM 2.5 level', 'PM 2.5 level num'],
      dtype='object')